In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn as sk
from sklearn.metrics import r2_score, mean_squared_error

### **Task:** Connect and ETL your locally cached/saved datasets

# Improving Data Quality and Predictive modelling:
## **Equifax Customer credit account data**

# Predictive modelling for KPI
The KPI or core metric we are interested in predicting is 'exp_flag'  [Equifax Performance Flag (1 = Bad, 0 = Good)]. This is a binary class outcome and best suited for classification modelling. The type of modelling will depend on the kind of data we can use.

**Objectives**

*   Obj1: Which classification methods can handle Nulls?
*   Obj2: Identify the fields that are most applicable to building a predictive model to classify the KPI
*   Obj3: What data quality issues do these fields present?

## **Task** : Does the target 'efx_gbflag' have balanced classes?

In [ ]:
df["efx_gbflag"].value_counts()

## **Task** : Filter dataframe to only see columns with over 25000 entries

In [ ]:
df_clean = df.apply(pd.to_numeric, errors='coerce')

In [ ]:
# counting number of values present in column,
# filtering to only see columns with over 25000 entries
count_table = df.count()
keep_cols_table = count_table[count_table > 25000]
keep_cols_table.sort_values(ascending=False)

#This results in 100 columns being filtered out for insufficient completeness

### **Task** : Create a dataframe without columns that have less than 50% complete entries

## Data preparation for Classification modelling

✅ Algorithms That Can Handle Nulls Natively or Gracefully
1. Decision Trees and Ensemble Methods (some implementations)
XGBoost:

✅ Natively handles missing values.

It learns optimal default directions when missing values are encountered.

xgboost.XGBClassifier in Python handles NaNs automatically.

Ref: Chen & Guestrin, 2016 – XGBoost: A Scalable Tree Boosting System

LightGBM:

✅ Handles NaNs internally during split finding.

Treats missing values as a separate split candidate.

Ref: Ke et al., 2017 – LightGBM: A Highly Efficient Gradient Boosting Decision Tree

CatBoost:

✅ Handles categorical and missing values without preprocessing.

Uses a data-driven approach for imputing during training.

Ref: Prokhorenkova et al., 2018 – CatBoost: Unbiased Boosting with Categorical Features

❌ Algorithms That Do Not Tolerate Nulls
Most of these will fail with NaNs unless you impute beforehand:

Logistic Regression

SVMs

k-Nearest Neighbours (KNN)

Naive Bayes

**Task:** what is the percentage of null values for each column in a dataframe, save those counts in a dataframe.

Some Classification approaches accept Nulls, most do not.






## **Task:** Evaluate the significance of relationships between the KPI and all the other input fields?

**Assessment:**
Nothing that seemingly strongly correlates, however the KPI is binary so correlation isnt as strong an indicator

Balance the sample sizes with SMOTE
- Handle potential missing values in the features before applying SMOTE
- SMOTE does not inherently handle missing values, so we need to impute them.
- A simple strategy is mean imputation, but you might want to explore more sophisticated methods.

In [ ]:
#!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = df_clean.drop('efx_gbflag', axis=1)
y = df_clean['efx_gbflag']

# Split data into training and testing sets before applying SMOTE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

for col in X_train.columns:
    if X_train[col].isnull().any():
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

# Note: SMOTE is applied only to the training data to prevent data leakage
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Original dataset shape %s" % y_train.value_counts())
print("Resampled dataset shape %s" % y_train_resampled.value_counts())

In [ ]:
X_train.isna().sum()

In [ ]:
#Alternative Imputer from sklearn that uses the mean
'''from sklearn.impute import SimpleImputer

#impute missing values in dataset with mean (BETTER WITH MODELLING THOUGH)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)


# Apply SMOTE to balance the classes
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_imputed_df, y)

# Check the new class distribution
print("Distribution before SMOTE:")
print(y.value_counts())
print("\nDistribution after SMOTE:")
print(y_resampled.value_counts())

# X_resampled and y_resampled now contain the balanced dataset
''';

In [ ]:
#build a classification model using the smoted data and evaluate

## **Task**: Build a classification model (logistic regression preferrably) using the smoted data and evaluate

### **Task**: Train a Decision Tree classification model

## **Task**: One more classification model, this time the ensemble Random Forest model

In [ ]:
# Save the random forest model

import pickle

# Save the trained Random Forest model
filename = 'random_forest_model.pkl'
pickle.dump(rf_model, open(filename, 'wb'))

# Download the model file
from google.colab import files
files.download(filename)


## ALTERNATIVELY for keeping it in the GCP ecosystem
1. Uploading Dataframe to GCP
2. Uploading Model to GCP

## UPLOAD DF TO GCP
2 Options
1. Big Query client, load table form dataframe
2. Big Frames package's to_gbq() streamlines process.

In [ ]:
#### OPTION 1 -- UPLOAD DF TO GCP ####

from google.cloud import bigquery

# Init BigQuery client
client = bigquery.Client()

# Define destination table
table_id = "your-project.your_dataset.your_table"

# Upload DataFrame
job = client.load_table_from_dataframe(df, table_id)
job.result()  # Wait for completion

print(f"Uploaded {job.output_rows} rows to {table_id}")

In [ ]:
#### OPTION 2 -- UPLOAD DF to GCP ####

import bigframes.pandas as bpd

bdf = bpd.from_pandas(df)
bdf.to_gbq("your-project.your_dataset.your_table", if_exists="replace")

## UPLOAD MODEL TO GCP
Two options

1.   Upload the trained model (.pkl, joblib) to Google Cloud Storage
2.   Register model with Vertex AI

In [ ]:
#UPLOAD TRAINED MODEL like .pkl and .joblib to GC Storage

from google.cloud import storage
import joblib

# Save model locally
joblib.dump(model, "model.pkl")

# Upload to GCS
client = storage.Client()
bucket = client.bucket("your-gcs-bucket")
blob = bucket.blob("models/model.pkl")
blob.upload_from_filename("model.pkl")

print("Model uploaded to GCS.")


In [ ]:
# REGISTER MODEL with VERTEX AI

from google.cloud import aiplatform

aiplatform.init(project="your-project", location="europe-west4")

model = aiplatform.Model.upload(
    display_name="my-model",
    artifact_uri="gs://your-gcs-bucket/models/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.0-24:latest"
)
